# 03 — Prepare and upload the Optimize dataset

**Foundry feature:** the **Optimize wizard's** dataset upload. The wizard has no column-mapping
step, so the file you upload has to already match its expected columns exactly. **Mode: CLI, then
Portal.**

In [ ]:
import json, subprocess, sys
from pathlib import Path

# Repo layout: this notebook lives in notebooks/, the pack lives in ../prompt-agent-optimizer-baselines
PACK_ROOT = Path("..").resolve() / "prompt-agent-optimizer-baselines"
AGENT_ID = "01-travel-approval-strict"          # <- the one case study every notebook in this series uses
AGENT_DIR = PACK_ROOT / AGENT_ID

assert AGENT_DIR.exists(), f"Can't find {AGENT_DIR} -- run this notebook from a checkout of the repo."

def run(cmd, cwd=PACK_ROOT):
    """Run a pack CLI tool and print its output, the way you would from a terminal."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

print(f"Pack root : {PACK_ROOT}")
print(f"Case study: {AGENT_ID}")

## CLI — strip authoring fields with `build_foundry_dataset.py`

The pack's raw `dataset/optimize.jsonl` rows carry extra authoring fields (`id`, `tags`) the wizard
doesn't expect. `_tools/build_foundry_dataset.py` produces a clean upload file with only `query` and
`ground_truth`.

In [ ]:
result = run(["python3", "_tools/build_foundry_dataset.py",
              "--agent", AGENT_ID, "--out", f"upload_{AGENT_ID[:2]}.jsonl"])

In [ ]:
upload_path = PACK_ROOT / f"upload_{AGENT_ID[:2]}.jsonl"
rows = [json.loads(l) for l in upload_path.read_text().splitlines() if l.strip()]
print(f"{len(rows)} rows ready for upload; each row has exactly these keys: {sorted(rows[0].keys())}")
print(json.dumps(rows[0], indent=2))

## The leakage guard, demonstrated

`build_foundry_dataset.py` refuses to process `holdout.jsonl` even if you point it there directly —
the holdout split exists specifically so the optimizer never sees it. Watch it refuse:

In [ ]:
# --agent always resolves to dataset/optimize.jsonl by construction, so the CLI flags alone can't
# point it at holdout.jsonl -- exercise the underlying guard directly instead, the same one
# build_foundry_dataset.py's main() runs against whatever --agent resolves to.
sys.path.insert(0, str(PACK_ROOT / "_tools"))
import build_foundry_dataset as bfd
try:
    bfd.load_rows(AGENT_DIR / "dataset" / "holdout.jsonl")
    print("NOT refused -- this would be a bug in the guard")
except SystemExit as e:
    print("Refused, as designed:\n ", e)

## Portal — upload

1. Open the agent from notebook 02, go to its **Optimize** tab.
2. Upload `upload_01.jsonl` (written above, at the pack root) as the evaluation dataset.
3. Leave it there — model selection and running the wizard is notebook 04.

## Next

Continue to **`04_run_optimize_wizard.ipynb`**.